# PI 7 Deel 3

Deze notebook bouwt, traint en evalueert een convolutionele Variational Autoencoder op 48×48 grayscale gezichtsbeelden en visualiseert reconstructies, de 2D latente ruimte en gegenereerde rasterbeelden.
Data wordt geladen via torchvision ImageFolder met preprocessing; training optimaliseert gewogen MSE plus KL-term (Adam + scheduler) en rapporteert train/val/test prestaties.

## Imports en plotting instellingen

Korte samenvatting:
- Importeert PyTorch, torchvision, matplotlib en numpy.
- Stelt handige aliassen in (nn, F) en zorgt dat matplotlib hoge DPI gebruikt voor scherpe figuren.

Wat de code doet:
- Laadt torch, torch.nn en functionaliteiten.
- Haalt ImageFolder, DataLoader en transforms uit torchvision binnen.
- Configureert matplotlib (figure.dpi, savefig.dpi, image.interpolation).

In [ ]:
import torch
from torch import nn
from torch.nn import functional as F
from torchvision.datasets import ImageFolder
from torch.utils.data import DataLoader
import torchvision.transforms as transforms
from torch.utils.data import random_split
import matplotlib.pyplot as plt
import numpy as np
plt.rcParams['figure.dpi'] = 300
plt.rcParams['savefig.dpi'] = 300
plt.rcParams['image.interpolation'] = 'bicubic'


## Encoder en Decoder
Korte samenvatting:
- Definieert convolutionele Encoder en transposed-conv / upsample Decoder voor de auto-encoder.

Wat de code doet:
- Encoder: meerdere Conv2d -> BatchNorm -> ReLU lagen die een 48x48 grayscale afbeelding stapgewijs verlagen naar een laag-dimensionale feature-map met z_size kanalen.
- Decoder: ConvTranspose2d lagen om de latent tensor terug omhoog te brengen, gevolgd door Upsample naar (48,48) en een eindconv naar 1 kanaal.
- Forward methods geven respectievelijk feature-map (encoder) en gereconstrueerde afbeelding (decoder).

In [ ]:
class Encoder(nn.Module):
    def __init__(self, z_size=2):
        super().__init__()
        self.z_size = z_size
        self.net = nn.Sequential(
            # 32 -> 16
            nn.Conv2d(1, 64, kernel_size=3, stride=2, padding=1, bias=False),
            nn.BatchNorm2d(64, affine=True),
            nn.ReLU(inplace=True),
            # 16 -> 8
            nn.Conv2d(64, 128, kernel_size=3, stride=2, padding=1, bias=False),
            nn.BatchNorm2d(128, affine=True),
            nn.ReLU(inplace=True),
            # 8 -> 4
            nn.Conv2d(128, 256, kernel_size=3, stride=2, padding=1, bias=False),
            nn.BatchNorm2d(256, affine=True),
            nn.ReLU(inplace=True),
            # 4 -> 1
            nn.Conv2d(256, z_size, kernel_size=3, stride=2, padding=0, bias=False),
        )
    
    def forward(self, X: torch.Tensor):
        return self.net(X)

class Decoder(nn.Module):
    def __init__(self, z_size=2):
        super().__init__()
        self.z_size = z_size
        self.deconv_layers = nn.Sequential(
            # 1 -> 4
            nn.ConvTranspose2d(z_size, 256, kernel_size=4, stride=1, padding=0, bias=False),
            nn.BatchNorm2d(256, affine=True),
            nn.ReLU(inplace=True),
            # 4 -> 8
            nn.ConvTranspose2d(256, 128, kernel_size=4, stride=2, padding=1, bias=False),
            nn.BatchNorm2d(128, affine=True),
            nn.ReLU(inplace=True),
            # 8 -> 16
            nn.ConvTranspose2d(128, 64, kernel_size=4, stride=2, padding=1, bias=False),
            nn.BatchNorm2d(64, affine=True),
            nn.ReLU(inplace=True),
            # 16 -> 32
            nn.ConvTranspose2d(64, 32, kernel_size=4, stride=2, padding=1, bias=False),
            nn.BatchNorm2d(32, affine=True),
            nn.ReLU(inplace=True),
        )
        self.upsample = nn.Upsample(size=(48, 48), mode='bilinear', align_corners=False)
        self.final_conv = nn.Conv2d(32, 1, kernel_size=3, padding=1, bias=False)

    def forward(self, X: torch.Tensor):
        x = self.deconv_layers(X)
        x = self.upsample(x)
        x = self.final_conv(x)
        return x



## VAE klasse (encoder + decoder integratie)
Korte samenvatting:
- Wrapt Encoder en Decoder in een eenvoudige VAE: encode -> sample -> decode.

Wat de code doet:
- __init__: maakt een Encoder met output 2*z_size (mu en log_var) en een Decoder met z_size.
- encode: voert input door encoder, neemt globale spatial average en splitst in mu en log_var.
- decode: samplet z = mu + exp(0.5*log_var)*eps en vormt z naar (B, z_dim, 1, 1) voor decoder.
- forward: retourneert reconstructie y, latente z, en mu/log_var.

In [ ]:
class VAE(nn.Module):
    def __init__(self, z_size=2):
        super().__init__()
        self.z_size = z_size
        self.enc = Encoder(2*z_size)
        self.dec = Decoder(z_size)
    
    def encode(self, X: torch.Tensor):
        h = self.enc(X) # (batch_size, 2*z_dim, H, W)
        # collapse spatial dimensions to get a (batch_size, 2*z_dim) vector
        h = h.view(h.size(0), h.size(1), -1).mean(dim=2)  # global spatial average
        mu = h[:, :self.z_size]       # (batch_size, z_dim)
        log_var = h[:, self.z_size:]  # (batch_size, z_dim)
        return mu, log_var
    
    def decode(self, mu: torch.Tensor, log_var: torch.Tensor):
        eps = torch.randn(mu.shape[0], self.z_size).to(mu.device) # random eps \sim N(0, I)
        z = mu + torch.exp(0.5 * log_var) * eps
        z = z.reshape(*z.shape, 1, 1) # (batch_size, z_dim, 1, 1)
        y = self.dec(z)
        return y, z
    
    def forward(self, X: torch.Tensor):
        mu, log_var = self.encode(X)
        y, z = self.decode(mu, log_var)
        return y, z, mu, log_var

## Data transforms en dataset laden
Korte samenvatting:
- Definieert preprocessing pipeline en laadt train/test mappen met ImageFolder; splitst train in train/val.

Wat de code doet:
- Transforms: converteert naar grayscale, resize naar 48x48, tensor, en normaliseert naar [-1,1] (mean=0.5, std=0.5).
- Laadt ImageFolder vanuit 'Data/train' en 'Data/test'.
- Maakt een train/validation split (80/20) met random_split.
- Bepaalt emotion_labels uit de class-folders en print deze.

In [ ]:
transform = transforms.Compose([
    transforms.Grayscale(),
    transforms.Resize((48, 48)),
    transforms.ToTensor(),
    transforms.Normalize([0.5], [0.5])
])


train_data = ImageFolder(root='Data/train', transform=transform)
test_data = ImageFolder(root='Data/test', transform=transform)

train_size = int(0.8 * len(train_data))
val_size = len(train_data) - train_size
train_subset, val_subset = random_split(train_data, [train_size, val_size])

emotion_labels = train_data.classes
print("Emotielabels:", emotion_labels)

## Trainingsinstellingen en apparaatkeuze
Korte samenvatting:
- Stelt hyperparameters voor training in en kiest CPU/GPU.

Wat de code doet:
- Definieert EPOCHS, BATCH_SIZE, LEARNING_RATE, gamma voor LR-scheduler.
- Zet Z_DIM en LAMBDA_REC_ERR (weegt reconstructieverlies).
- Controleert en print of CUDA beschikbaar is; stelt DEVICE in op cuda of cpu.

In [ ]:
# Training constants
EPOCHS = 20
BATCH_SIZE = 256
LEARNING_RATE = 0.003
LEARNING_RATE_GAMMA = 0.9

# Model constants
Z_DIM = 2
LAMBDA_REC_ERR = 100

# check for possible GPU usage
DEVICE = torch.device('cuda') if torch.cuda.is_available() else torch.device('cpu')
print("Using device:", DEVICE)

## Training, validatie en test loop
Korte samenvatting:
- Bouwt dataloaders, model, optimizer en trainer loop met KL-regularisatie en gewogen MSE reconstructieverlies.

Wat de code doet (kort):
- Maakt DataLoader objecten voor train/val/test.
- Initialiseert VAE, Adam optimizer en ExponentialLR scheduler.
- Gebruikt MSELoss als reconstructiecriterium.
- Voor elk epoch: train-loop met loss = LAMBDA_REC_ERR * MSE + KL, backward en optimizer.step(); scheduler.step() na epoch.
- Valideren met torch.no_grad(), opslaan van train/val historie.
- Na training: evaluatie op testset en print average test loss.

In [ ]:
train_loader = DataLoader(train_subset, batch_size=BATCH_SIZE, shuffle=True)
val_loader   = DataLoader(val_subset,   batch_size=BATCH_SIZE, shuffle=False)
test_loader  = DataLoader(test_data,    batch_size=BATCH_SIZE, shuffle=False)

generator = VAE(z_size=Z_DIM).to(DEVICE)
optimizer = torch.optim.Adam(generator.parameters(), lr=LEARNING_RATE)
scheduler = torch.optim.lr_scheduler.ExponentialLR(optimizer, gamma=LEARNING_RATE_GAMMA)

criterion = nn.MSELoss()

train_history = []
val_history = []

for epoch in range(EPOCHS):
    batch_losses = []

    generator.train()
    for images, _ in train_loader:
        images = images.to(DEVICE)
        preds, _, mu, log_var = generator(images)

        kl_reg = torch.mean(-0.5 * torch.sum(1 + log_var - mu.pow(2) - log_var.exp(), dim=1))

        rec_error = LAMBDA_REC_ERR * criterion(preds, images)

        loss = rec_error + kl_reg
        batch_losses.append(loss.item())

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

    scheduler.step()
    train_history.append(sum(batch_losses) / len(batch_losses))

    generator.eval()
    val_losses = []
    with torch.no_grad():
        for X_val, _ in val_loader:
            X_val = X_val.to(DEVICE)
            preds, _, mu, log_var = generator(X_val)
            kl_reg = torch.mean(-0.5 * torch.sum(1 + log_var - mu.pow(2) - log_var.exp(), dim=1))
            rec_error = LAMBDA_REC_ERR * criterion(preds, X_val)
            val_losses.append((rec_error + kl_reg).item())

    val_history.append(sum(val_losses) / len(val_losses))

    print(f"Epoch {epoch+1}/{EPOCHS}, Train Loss: {train_history[-1]:.4f}, Val Loss: {val_history[-1]:.4f}")

generator.eval()
total_loss = 0
with torch.no_grad():
    for X_test, _ in test_loader:
        X_test = X_test.to(DEVICE)
        preds, _, mu, log_var = generator(X_test)
        kl_reg = torch.mean(-0.5 * torch.sum(1 + log_var - mu.pow(2) - log_var.exp(), dim=1))
        rec_error = LAMBDA_REC_ERR * criterion(preds, X_test)
        total_loss += (rec_error + kl_reg).item()

avg_test_loss = total_loss / len(test_loader)
print(f"Average Test Loss: {avg_test_loss:.4f}")


## Visualisatie: Latent ruimte en verliescurves
Korte samenvatting:
- Visualiseert 2D latent embeddings en training vs validatieverlies.

Wat de code doet:
- Loopt door test_loader, verzamelt z (gesampled latents) en bijbehorende labels.
- Maakt een scatterplot van z[:,0] vs z[:,1], kleurt punten naar emotion_labels en toont legenda.
- Teken daarnaast de train- en val-loss curves om trainingstrend te beoordelen.

In [ ]:
z_list, y_list = [], []
with torch.no_grad():
    for X_test, y_batch in test_loader:
        X_test = X_test.to(DEVICE)
        _, z, _, _ = generator(X_test)
        z_list.append(z.squeeze().cpu())
        y_list.append(y_batch)

z = torch.cat(z_list, dim=0)
y_test = torch.cat(y_list, dim=0)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(20, 10), dpi=300)

sc = ax1.scatter(z[:, 0], z[:, 1], c=y_test, s=20, cmap='tab10', alpha=0.6)
handles, _ = sc.legend_elements()
ax1.legend(handles, emotion_labels, title="Emotions", loc="best", fontsize=10)
ax1.set_title("Latent space (z)", fontsize=12)
ax1.grid(True, alpha=0.3)

ax2.plot(train_history, label="train", linewidth=2)
ax2.plot(val_history, label="val", linewidth=2)
ax2.set_title("Training vs Validation Loss", fontsize=12)
ax2.legend(fontsize=10)
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## Visualisatie: Origineel vs reconstructie
Korte samenvatting:
- Toont één voorbeeldafbeelding uit de testbatch naast de reconstructie van de VAE.

Wat de code doet:
- Haalt een batch uit test_loader, kiest index idx (hier 99).
- Voert het voorbeeld door het model, past sigmoid toe op output voor weergave.
- Plot originele en gereconstrueerde afbeelding naast elkaar zonder assen.

In [ ]:
X_test_batch, _ = next(iter(test_loader))
idx = 99
X_sample = X_test_batch[idx].unsqueeze(0).to(DEVICE)

# Pass through VAE
y_pred, _, _, _ = generator(X_sample)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 6), dpi=300)
ax1.set_title("Original X")
ax2.set_title("Reconstruction X~")

# Move to CPU and convert to numpy for plotting
orig = X_test_batch[idx].cpu().squeeze().numpy()
recon = F.sigmoid(y_pred).cpu().squeeze().detach().numpy()

ax1.imshow(orig, cmap='gray', interpolation='bicubic')
ax2.imshow(recon, cmap='gray', interpolation='bicubic')

# Verwijder de assen voor een cleaner look
ax1.axis('off')
ax2.axis('off')

plt.tight_layout()
plt.show()

## Latente rastergeneratie (model decoder sampling)
Korte samenvatting:
- Genereert een rooster van afbeeldingen door een 2D-grid in latent-ruimte door de decoder te sturen.

Wat de code doet:
- Maakt z1, z2 ranges (bijv. -2..2) en combineert tot een grid van (h*w, 2, 1, 1).
- Zet grid naar GPU, decodeert met generator.dec en past sigmoid voor zichtbaarheid.
- Vormt de resultaten om naar een grote beeldmat en toont dit als één grote montage.

In [ ]:
h, w = 15, 15
z1_range = torch.linspace(-2, 2, h)
z2_range = torch.linspace(-2, 2, w)

z1, z2 = torch.meshgrid(z1_range, z2_range, indexing='ij')
zz = torch.stack([z1, z2], dim=-1).reshape(-1, 2, 1, 1).to('cuda')

with torch.no_grad():
    y_preds = torch.sigmoid(generator.dec(zz)).cpu()

faces = y_preds.squeeze(1)

faces_grid = faces.reshape(h, w, 48, 48)
faces_grid = faces_grid.permute(0, 2, 1, 3)
faces_grid = faces_grid.reshape(h*48, w*48)

plt.figure(figsize=(8, 8))
plt.imshow(faces_grid, cmap='gray', interpolation='bicubic')
plt.axis('off')
plt.xlabel(r'$Z_1$')
plt.ylabel(r'$Z_2$')
plt.show()